# Comprehensive GeoAI Analytics: Deep Dive
Focusing strictly on massive, high-resolution Agronomic and Spectral EDA.


In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm
from collections import Counter
from scipy.ndimage import label

warnings.filterwarnings('ignore')
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.append(os.getcwd())

from src.data_loading import DataManager
manager = DataManager()

config = manager.config
metadata = manager.load_metadata()
patch_ids = manager.get_patch_ids(metadata)
s2_dir = config['paths']['s2_dir']
target_dir = config['paths']['annotations_dir']
CLASS_NAMES = config['classes']

cmap = cm.get_cmap('tab20', 20)
COLORS_BY_ID = {i: cmap(i) for i in range(20)}
COLORS_BY_NAME = {CLASS_NAMES[i]: cmap(i) for i in range(20)}
sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 14}) # Large fonts for big figures


## 1. Multi-Spectral Optical Visualization
Analyzing different spectral composites (True Color, False Color, SWIR) on a massive high-resolution grid.


In [ ]:
sample_pid = patch_ids[0]
s2_sample, target_sample = manager.load_patch_data(sample_pid)

true_color = np.clip(s2_sample[20, 1:4, :, :].transpose(1, 2, 0) / 3000.0, 0, 1)
false_color = np.clip(s2_sample[20, [6, 2, 1], :, :].transpose(1, 2, 0) / 3000.0, 0, 1)
swir_color = np.clip(s2_sample[20, [9, 6, 2], :, :].transpose(1, 2, 0) / 4000.0, 0, 1)

# MASSIVE 2x2 FIGURE (20x20 inches)
fig, axes = plt.subplots(2, 2, figsize=(20, 20))

axes[0, 0].imshow(true_color)
axes[0, 0].set_title('True Color (RGB) - Human Vision', fontsize=18, fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(false_color)
axes[0, 1].set_title('False Color (NIR) - Biomass Indicator', fontsize=18, fontweight='bold')
axes[0, 1].axis('off')

axes[1, 0].imshow(swir_color)
axes[1, 0].set_title('SWIR Composite - Moisture Content', fontsize=18, fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(target_sample[0], cmap=cmap, vmin=0, vmax=19)
axes[1, 1].set_title('Ground Truth Annotated Labels', fontsize=18, fontweight='bold')
axes[1, 1].axis('off')

plt.suptitle(f'Comprehensive Multi-Spectral View of Farm Patch {sample_pid}', fontsize=24, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 2. Landscape Analytics: Area Distribution
Calculating total area per crop across the entire dataset.


In [ ]:
print('Scanning dataset for global spatial distributions...')
class_counts = Counter()
field_sizes = []

for pid in patch_ids:
    target = np.load(os.path.join(target_dir, f'TARGET_{pid}.npy'))
    u, c = np.unique(target, return_counts=True)
    for val, count in zip(u, c): class_counts[val] += count
        
    # Connected Component Analysis for Field Size (Class 2 - Winter Wheat)
    lbl, num_features = label(target[0] == 2)
    for i in range(1, num_features + 1):
        field_sizes.append(np.sum(lbl == i))

df_dist = pd.DataFrame.from_dict(class_counts, orient='index', columns=['Pixels']).reset_index()
df_dist = df_dist[df_dist['index'] != 0].copy()
df_dist['Crop'] = df_dist['index'].map(CLASS_NAMES)
df_dist = df_dist.sort_values('Pixels', ascending=False)

# MASSIVE SINGLE PLOT
plt.figure(figsize=(20, 10))
sns.barplot(data=df_dist, x='Crop', y='Pixels', palette=COLORS_BY_NAME, hue='Crop', legend=False)
plt.xticks(rotation=45, ha='right', fontsize=16)
plt.yticks(fontsize=14)
plt.yscale('log')
plt.title('Global Crop Area Distribution (Log Scale)', fontsize=22, fontweight='bold')
plt.ylabel('Total Area (Pixels)', fontsize=18)
plt.xlabel('')
sns.despine()
plt.tight_layout()
plt.show()


## 3. Landscape Analytics: Field Sizes
Estimating the physical size distribution of Winter Wheat fields.


In [ ]:
# MASSIVE SINGLE PLOT
plt.figure(figsize=(18, 8))
sns.histplot(field_sizes, bins=40, color=COLORS_BY_ID[2], kde=True)
plt.title('Winter Wheat: Individual Field Size Distribution', fontsize=22, fontweight='bold')
plt.xlabel('Field Size (Pixels)', fontsize=18)
plt.ylabel('Number of Fields', fontsize=18)
sns.despine()
plt.show()


## 4. Spatial Diversity & Co-occurrence
How do crops interact spatially? Who shares patches?


In [ ]:
diversity_scores = []
co_matrix = np.zeros((20, 20))

for pid in patch_ids:
    target = np.load(os.path.join(target_dir, f'TARGET_{pid}.npy'))
    unique_crops = np.unique(target[target != 0])
    diversity_scores.append(len(unique_crops))
    
    for i in unique_crops:
        for j in unique_crops:
            if i != j:
                co_matrix[i, j] += 1

# MASSIVE SINGLE PLOT FOR DIVERSITY
plt.figure(figsize=(18, 8))
sns.histplot(diversity_scores, bins=np.arange(0.5, 12.5, 1), color='teal')
plt.title('Crop Diversity: Unique Crops per 1.2km Patch', fontsize=22, fontweight='bold')
plt.xlabel('Number of Unique Crop Types per Patch', fontsize=18)
plt.ylabel('Frequency (Number of Patches)', fontsize=18)
plt.xticks(range(1, 13), fontsize=16)
sns.despine()
plt.show()


## 5. Co-occurrence Matrix
Mapping which crops are most frequently planted together.


In [ ]:
active_classes = df_dist['index'].head(12).values # Top 12
df_co = pd.DataFrame(co_matrix[np.ix_(active_classes, active_classes)], 
                     index=[CLASS_NAMES[c] for c in active_classes],
                     columns=[CLASS_NAMES[c] for c in active_classes])

# MASSIVE HEATMAP
plt.figure(figsize=(18, 14))
sns.heatmap(df_co, annot=True, fmt='.0f', cmap='YlGnBu', annot_kws={'size': 16})
plt.title('Crop Spatial Co-occurrence Matrix (Top 12)', fontsize=22, fontweight='bold')
plt.xticks(fontsize=14)
plt.yticks(fontsize=14, rotation=0)
plt.show()


## 6. Spectral Physics & Signatures
Analyzing the distinct signatures of crops across 10 Sentinel-2 bands.


In [ ]:
print('Extracting high-dimensional pixel data for physics analysis...')
s2_flat_list, target_flat_list = [], []

for p in patch_ids[:20]:
    s, t = manager.load_patch_data(p)
    s2_flat_list.append(s.mean(axis=0).reshape(10, -1).T)
    target_flat_list.append(t.flatten())
    
X = np.vstack(s2_flat_list)
y = np.concatenate(target_flat_list)
df_physics = pd.DataFrame(X, columns=config['dataset']['bands'])

# MASSIVE BAND CORRELATION HEATMAP
plt.figure(figsize=(16, 12))
sns.heatmap(df_physics.corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True, annot_kws={'size': 16})
plt.title('Sentinel-2 Band Correlation (Collinearity)', fontsize=22, fontweight='bold')
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.show()


## 7. Mean Spectral Reflectance Signatures
How do different crops reflect light across the spectrum?


In [ ]:
# MASSIVE SIGNATURE PLOT
plt.figure(figsize=(20, 10))
for c in [1, 2, 5, 3, 4]: # Meadow, Wheat, Rapeseed, Corn, Barley
    mean_sig = X[y == c].mean(axis=0)
    plt.plot(config['dataset']['bands'], mean_sig, label=CLASS_NAMES[c], color=COLORS_BY_ID[c], lw=5)

plt.title('Mean Spectral Reflectance Signatures (Physics of Light)', fontsize=24, fontweight='bold')
plt.xlabel('Sentinel-2 Spectral Band', fontsize=20)
plt.ylabel('Mean Reflectance Intensity', fontsize=20)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.legend(fontsize=18)
sns.despine()
plt.show()


## 8. Advanced Temporal Indices (NDVI - Biomass)
Mapping the growth cycles of crops across the 46 time steps.


In [ ]:
print('Processing 4D arrays for temporal agronomic indices...')
ndvi_series = {c: [] for c in range(1, 20)}
ndwi_series = {c: [] for c in range(1, 20)}

for p in patch_ids[:30]:
    s, t = manager.load_patch_data(p)
    ndvi = (s[:, 6] - s[:, 2]) / (s[:, 6] + s[:, 2] + 1e-8)
    ndwi = (s[:, 1] - s[:, 8]) / (s[:, 1] + s[:, 8] + 1e-8)
    for c in range(1, 20):
        if np.any(t[0] == c):
            ndvi_series[c].append(ndvi[:, t[0] == c].mean(axis=1))
            ndwi_series[c].append(ndwi[:, t[0] == c].mean(axis=1))

target_crops = [1, 2, 3, 5] # Meadow, Wheat, Corn, Rapeseed

# MASSIVE NDVI PLOT
plt.figure(figsize=(22, 10))
for c in target_crops:
    if ndvi_series[c]:
        mean_ndvi = np.mean(np.vstack(ndvi_series[c]), axis=0)
        plt.plot(mean_ndvi, label=CLASS_NAMES[c], color=COLORS_BY_ID[c], lw=5)

plt.title('NDVI Trajectories (Biomass/Vegetation Growth Cycles)', fontsize=24, fontweight='bold')
plt.xlabel('Time Step (0 to 45)', fontsize=20)
plt.ylabel('Mean NDVI', fontsize=20)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.legend(fontsize=18)
sns.despine()
plt.show()


## 9. Advanced Temporal Indices (NDWI - Moisture)
Mapping the moisture and water content cycles of the same crops.


In [ ]:
# MASSIVE NDWI PLOT
plt.figure(figsize=(22, 10))
for c in target_crops:
    if ndwi_series[c]:
        mean_ndwi = np.mean(np.vstack(ndwi_series[c]), axis=0)
        plt.plot(mean_ndwi, label=CLASS_NAMES[c], color=COLORS_BY_ID[c], lw=5)

plt.title('NDWI Trajectories (Crop Water/Moisture Content)', fontsize=24, fontweight='bold')
plt.xlabel('Time Step (0 to 45)', fontsize=20)
plt.ylabel('Mean NDWI', fontsize=20)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.legend(fontsize=18)
sns.despine()
plt.show()
